# Figure 2 Dashboard — VPD Step Response Parameter Explorer

Explore how different parameter groups affect the stomatal response to a VPD step (RH: 80% → 60%).

**Dashboards:**
1. **Hydraulics** — tissue resistances, elastic moduli, conductance gain
2. **Vstress → ΔPg curve** — stress-sensing sigmoid
3. **ABA Biosynthesis** — synthesis/catabolism kinetics
4. **ABA Signaling** — receptor-kinase cascade
5. **Electrophysiology** — ion channels, pumps

---

**Paper.** This notebook reproduces **Figure 2** of:

> Desai, S. A., & Stroock, A. D. (2025). *Abscisic acid-mediated water stress regulation can mechanistically explain oscillations and water stress memory in stomatal conductance.* bioRxiv. https://doi.org/10.64898/2025.12.26.696581

Hosted, no-install versions of all three dashboards: https://desai-sahil.github.io/stomatal-conductance-model/


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import io, contextlib, warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd

import panel as pn
pn.extension("bokeh")

print('Imports loaded (matplotlib + Panel/Bokeh).')


## 2. Model Parameters

In [2]:
def read_parameters_wt_Merilo_2018():
    """
    Returns a dictionary of model parameters for the coupled hydropassive-
    hydroactive stomatal conductance model (Merilo 2018 wild-type Arabidopsis).
    
    Includes: tissue hydraulics, ABA biosynthesis/catabolism, ABA signaling
    cascade, guard cell electrophysiology, and simulation timing.
    """
    hyd = {}

    # =========================================================================
    # 1. TISSUE VOLUMES & TURGOR PRESSURES
    # =========================================================================
    # Symplastic water volumes (mmol H₂O / m² leaf area)
    hyd["Vg0"] = 100.0                     # guard cell
    hyd["Vs0"] = 100.0                     # subsidiary cell
    hyd["Vm0"] = 1978.0                    # mesophyll

    hyd["stomatal_density"] = 150e6        # stomata / m² leaf

    # Bulk elastic moduli (MPa) — tissue stiffness
    hyd["epsilong"] = 3.6                  # guard cell
    hyd["epsilons"] = 0.5                  # subsidiary cell
    hyd["epsilonm"] = 4.0                  # mesophyll

    # Initial turgor pressures (MPa)
    hyd["Pg0"] = 2.5                       # guard cell
    hyd["Ps0"] = 1.0                       # subsidiary cell
    hyd["Pm0"] = 2.2                       # mesophyll

    # Initial osmotic pressures (MPa); offset from turgor by 0.1
    hyd["pig0"] = hyd["Pg0"] + 0.1
    hyd["pis0"] = hyd["Ps0"] + 0.1
    hyd["pim0"] = hyd["Pm0"] + 0.1

    # Initial turgor perturbations (MPa)
    hyd["DeltaPim"] = 0.0
    hyd["DeltaPis"] = 0.0
    hyd["DeltaPig"] = 0.0

    # =========================================================================
    # 2. GAS CONSTANTS & HYDRAULIC TRANSPORT
    # =========================================================================
    hyd["R"]    = 8.314                    # universal gas constant (J/mol/K)
    hyd["T"]    = 308.0                    # temperature (K) ≈ 35°C
    hyd["nu_w"] = 1.8e-5                   # molar volume of water (m³/mol)
    hyd["RT_by_nu_by1e6"] = (hyd["R"] * hyd["T"] / hyd["nu_w"]) / 1e6

    # Stomatal conductance equation:  gs = Xi * max(Pg - m*Ps, 0)
    hyd["Xi"] = 295.0                      # conductance gain (mmol/m²/s/MPa)
    hyd["m"]  = 1.71                       # mechanical advantage of subsidiary cells

    # Hydraulic resistances (MPa·s·m²/mmol) — flow = ΔΨ / R
    hyd["Rxm"] = 0.018116                  # xylem → mesophyll
    hyd["Rms"] = 5.0                       # mesophyll → subsidiary cell
    hyd["Rsg"] = 10.0                      # subsidiary cell → guard cell

    # =========================================================================
    # 3. CONDUCTANCE BASELINES & ATMOSPHERIC CONDITIONS
    # =========================================================================
    # Stomatal and cuticular conductance coefficients (mmol/m²/s)
    hyd["gg1"] = 2.9                       # g_g¹ — guard cell (stomatal) pathway
    hyd["ge1"] = 2.9                       # g_e¹ — epidermal (cuticular) pathway
    hyd["gg2"] = 1.0                       # g_g² — guard cell VPD sensitivity
    hyd["ge2"] = 10.0                      # g_e² — epidermal VPD sensitivity

    # Saturation vapor pressure and atmospheric pressure
    hyd["psat"]      = 0.00563             # saturation vapor pressure (MPa) at T
    hyd["patm"]      = 0.101               # atmospheric pressure (MPa)
    hyd["psat_patm"] = hyd["psat"] / hyd["patm"]

    # =========================================================================
    # 4. HUMIDITY PROTOCOL (VPD STEP)
    # =========================================================================
    hyd["RH1"] = 80.0                      # initial RH (%)
    hyd["RH2"] = 60.0                      # post-step RH (%)
    hyd["VPD1"] = hyd["psat"] * (1 - hyd["RH1"] / 100)
    hyd["VPD2"] = hyd["psat"] * (1 - hyd["RH2"] / 100)
    hyd["psix_at_RH_step"] = -0.1          # xylem water potential at step (MPa)

    # =========================================================================
    # 5. ABA BIOSYNTHESIS & CATABOLISM
    #    Autoregulatory motif: NCED (positive fb) + CYP707A/A8H (negative fb)
    # =========================================================================
    # Genetic pre-factors (1.0 = wild-type; set to 0 for knockouts)
    hyd["thetaABA"] = 1.0                  # θ_NCED — synthesis (NCED3/5)
    hyd["thetaA8H"] = 1.0                  # θ_A8H  — catabolism (CYP707A)

    hyd["n"]    = 3                        # Hill coefficient (stress cooperativity)
    hyd["kcat"] = 0.25                     # γ_A: A8H catalytic rate (1/s)
    hyd["K2"]   = 2000.0                   # K_d: A8H Michaelis constant (nM)
    hyd["gammac"] = 32 * 9.6e-5            # γ_H: A8H protein decay rate (1/s) ≈ 3.072e-3

    # Half-saturation ratios: K₊ = λ₁·K_d,  K₋ = λ₃·K_d
    hyd["lambda1"] = 0.78
    hyd["lambda3"] = 0.78
    hyd["K1"] = hyd["lambda1"] * hyd["K2"]  # K₊ (nM) — NCED half-saturation
    hyd["K3"] = hyd["lambda3"] * hyd["K2"]  # K₋ (nM) — CYP707A half-saturation

    # Dimensionless rate ratios (φ) → derive dimensional rates (V)
    hyd["phi1"] = 0.072                    # stress synthesis scaling
    hyd["phi2"] = 0.55                     # positive feedback strength
    hyd["phi3"] = 20.0                     # catabolism scaling

    # Derived maximal rates (nM/s)
    hyd["Vminus"]  = (hyd["K2"] * hyd["gammac"]**2 * hyd["phi3"]) / hyd["kcat"]
    hyd["Vplus"]   = hyd["phi2"] * hyd["kcat"] * hyd["Vminus"] / hyd["gammac"]
    hyd["Vstress"] = hyd["phi1"] * hyd["kcat"] * hyd["Vminus"] / hyd["gammac"]

    # =========================================================================
    # 6. ABA SIGNALING CASCADE
    #    Bicyclic futile cycle: MAP3K → OST1 → SLAC1
    #    with PP2C dephosphorylation, inhibited by ABA:PYR complex
    # =========================================================================
    # Total protein concentrations (nM)
    hyd["OST1T"]  = 1e3
    hyd["SLAC1T"] = 1e3
    hyd["MAP3KT"] = 1e3
    hyd["PP2CT"]  = 1e3
    hyd["PYRT"]   = 1e3

    # ABA-receptor binding
    hyd["KD"] = 1000.0                     # ABA–PYR dissociation constant (nM)
    hyd["KI"] = 30.0                       # PYR–PP2C inhibition constant (nM)

    # Phosphorylation / dephosphorylation rates (1/s)
    hyd["k1"] = 0.03                       # MAP3K → OST1 (phosphorylation)
    hyd["k2"] = 0.032772                   # PP2C ⊣ OST1 (dephosphorylation)
    hyd["k3"] = 3.75                       # OST1 → SLAC1 (phosphorylation)
    hyd["k4"] = 0.1386                     # PP2C ⊣ SLAC1 (dephosphorylation)

    # Michaelis constants for the four enzymatic steps (nM)
    hyd["Km1"] = 2.5e3
    hyd["Km2"] = 0.0897e3
    hyd["Km3"] = 18.93e3
    hyd["Km4"] = 0.597e3

    # =========================================================================
    # 7. GUARD CELL ELECTROPHYSIOLOGY
    #    GHK ion fluxes through SLAC1 (Cl⁻), GORK (K⁺ out), KAT1 (K⁺ in),
    #    and H⁺-ATPase proton pump
    # =========================================================================
    # Membrane properties
    hyd["Cm"]   = 1e-2                     # membrane capacitance (F/m²)
    hyd["area"] = 20e-10                   # guard cell membrane area (m²)
    hyd["vol_in"]  = hyd["Vg0"] * hyd["nu_w"] * 1e-3 / hyd["stomatal_density"]  # intracellular volume (m³)
    hyd["vol_out"] = 65e-15                # apoplastic volume (m³)
    hyd["F"] = 96485.3329                  # Faraday constant (C/mol)

    # GORK voltage gating
    hyd["delta_GORK"]         = 2.0        # voltage sensitivity parameter
    hyd["V1by2_Kconc_GORK"]   = 150.0      # half-activation K⁺ concentration (mM)

    # KAT1 voltage gating
    hyd["delta_KAT1"]  = 1.6              # voltage sensitivity parameter
    hyd["V1by2_KAT1"]  = -170e-3          # half-activation voltage (V)

    hyd["otherIons"] = 0.0                 # background ion contribution

    # Initial ion concentrations (mM)
    hyd["Clin0"]  = 260.0                  # Cl⁻ intracellular
    hyd["Clout0"] = 22.0                   # Cl⁻ apoplastic
    hyd["Kin0"]   = 210.0                  # K⁺  intracellular
    hyd["Kout0"]  = 20.0                   # K⁺  apoplastic

    # Ion valences
    hyd["zCl"] = -1
    hyd["zK"]  = 1

    # Membrane permeabilities (m/s)
    hyd["PK"]  = 5e-6                      # K⁺ permeability
    hyd["PCl"] = 1e-7                      # Cl⁻ permeability

    # Channel number scaling (relative to baseline)
    hyd["NGORK"]  = 1.0
    hyd["NKAT1"]  = 1.0
    hyd["NSLAC1"] = 1.0

    # Proton concentrations (derived from pH)
    hyd["pHin0"]  = 7.2
    hyd["pHout0"] = 6.5
    hyd["Hin0"]   = 10**(-hyd["pHin0"])  * 1e3   # intracellular H⁺ (mM)
    hyd["Hout0"]  = 10**(-hyd["pHout0"]) * 1e3   # apoplastic H⁺ (mM)
    hyd["zH"] = 1

    # H⁺-ATPase proton pump
    hyd["DeltaGATP"]      = -26000.0       # free energy of ATP hydrolysis (J/mol)
    hyd["E0pump"]         = 1.1033e6       # pump rate constant
    hyd["k1pump"]         = 0.55           # pump kinetic parameter 1
    hyd["k2pump"]         = 2.58e-4        # pump kinetic parameter 2
    hyd["V0Cl"]           = 9.25e4         # SLAC1 Cl⁻ flux scaling
    hyd["ABA_Hpump_Khalf"] = 50.0          # ABA half-inhibition of H⁺-pump (nM)
    hyd["ABA_Hpump_n"]    = 1.0            # Hill coefficient for ABA pump inhibition

    # =========================================================================
    # 8. SIMULATION TIMING
    # =========================================================================
    hyd["tstart"] = 0.0                    # simulation start (s)
    hyd["tstep1"] = 120000.0               # time of VPD step (s) = 2000 min
    hyd["tend"]   = 240000.0               # simulation end (s)   = 4000 min

    return hyd

print("Parameter function defined.")


Parameter function defined.


## 3. Helper Functions

In [3]:
# =============================================================================
# HELPER FUNCTIONS
# Organized by module: Environmental → Hydraulics → ABA → Signaling →
#                      Ion Channels → Membrane Transport
# =============================================================================

# ---- Environmental forcing (Figure 2: single VPD step) ----------------------

def f_RH(t, hyd):
    """Relative humidity protocol: step from RH1 → RH2 at t = tstep1."""
    return np.where(np.asarray(t) <= hyd["tstep1"], hyd["RH1"], hyd["RH2"])

def f_VPD(t, hyd):
    """Vapor pressure deficit corresponding to the RH step."""
    return np.where(np.asarray(t) <= hyd["tstep1"], hyd["VPD1"], hyd["VPD2"])

def f_psix(t, hyd):
    """Xylem water potential (MPa). Held constant for Figure 2."""
    return hyd["psix_at_RH_step"]

# ---- Hydraulic transport ----------------------------------------------------

def f_Rox(psix):
    """Overall xylem resistance as a function of xylem water potential.
    
    Empirical sigmoidal fit capturing vulnerability-curve–driven loss of
    xylem hydraulic conductance under drought (increasing resistance as
    psix becomes more negative).
    """
    a, b, c, k = 0.3664, 15.61, -0.7684, 2.958
    return 1.0 / (a + (k - a) / (1.0 + np.exp(-b * (psix - c))))

def f_P_from_psi(psi, epsilon, pi0, P0, DeltaPi_MPa):
    """Convert water potential perturbation (ψ) to turgor pressure (P).
    
    Uses the linearized relation:  P = P0 + ψ / (1 + π0/ε)
    where ε = bulk elastic modulus, π0 = initial osmotic pressure.
    Turgor is clamped to ≥ 0 (no negative turgor).
    """
    return np.maximum(P0 + (psi - DeltaPi_MPa) / (1.0 + pi0 / epsilon), 0.0)

def f_SSC_apoplast_potential(psim, psig, psis, gs, gmH2O, pa, hyd, geH2O, ggH2O):
    """Steady-state substomatal cavity (SSC) apoplast water potential.
    
    Balances water inflow from mesophyll, subsidiary, and guard cells against
    evaporative loss through stomata. This is the node potential at the
    evaporation site in the hydraulic circuit.
    """
    return (psim * gmH2O + psis * geH2O + psig * ggH2O
            - gs * hyd["RT_by_nu_by1e6"] * (hyd["psat"] - pa) / hyd["psat"]) \
            / (gmH2O + geH2O + ggH2O + gs)

# ---- ABA biosynthesis & catabolism ------------------------------------------

def f_Vstress(DeltaPg, hyd):
    """Stress-responsive ABA synthesis rate as a Hill function of turgor loss.
    
    V_stress(ΔPg) = V_stress · |ΔPg|^n / (K_half + |ΔPg|^n)
    
    Uses custom vstress_exponent and vstress_khalf if provided via
    dashboard overrides, otherwise defaults to n=3.25 and K_half=1.4.
    """
    abs_dpg = np.abs(DeltaPg)
    n_hill = hyd.get("vstress_exponent", 3.25)
    k_half = hyd.get("vstress_khalf", 1.4)
    return np.abs(hyd["Vstress"] * abs_dpg**n_hill / (k_half + abs_dpg**n_hill))

# ---- ABA signaling cascade -------------------------------------------------

def f_solve_ABA_PYR_quadratic(ABA, hyd):
    """Solve for free PYR receptor concentration given [ABA].
    
    ABA binds PYR receptors; the ABA:PYR complex inhibits PP2C. This
    solves the quadratic mass-balance equation for [PYR_free], accounting
    for total PYR, dissociation constant K_D, and inhibition constant K_I.
    Returns the physically meaningful (positive) root.
    """
    theta_PYR = 1.0 + hyd["KD"] / ABA
    b = hyd["KI"] + (hyd["PP2CT"] - hyd["PYRT"]) / theta_PYR
    c = -hyd["KI"] * hyd["PYRT"] / theta_PYR
    return (-b + np.sqrt(b**2 - 4.0 * c)) / 2.0

def f_SLAC1_conc_non_dimensional(Oo, So, hyd):
    """Unphosphorylated SLAC1 fraction from conservation law.
    
    Given OST1* fraction (Oo) and SLAC1* fraction (So), computes the
    free (unphosphorylated) SLAC1 fraction S using the total protein
    conservation: S + So + [enzyme-bound So] = 1.
    """
    K4p   = hyd["K4_nd"] * (1.0 + Oo / hyd["K2_nd"])
    term3 = (hyd["PP2CT"] / hyd["SLAC1T"]) * So / (K4p + So)
    return (1.0 - So - term3) / (1.0 + (hyd["OST1T"] / hyd["SLAC1T"]) * (Oo / hyd["K3_nd"]))

def f_OST1_roots_from_quadratic_non_dimensional(Oo, So, hyd):
    """Unphosphorylated OST1 fraction from conservation law.
    
    Solves the quadratic arising from OST1 total conservation:
    O + Oo + [enzyme-bound fractions] = 1.
    Returns the positive root (clamped to ≥ 0).
    """
    S     = f_SLAC1_conc_non_dimensional(Oo, So, hyd)
    theta = 1.0 - Oo * (1.0 + S / hyd["K3_nd"])
    b     = hyd["epsilon1"] + hyd["K1_nd"] - theta
    c     = -theta * hyd["K1_nd"]
    return max((-b + np.sqrt(b**2 - 4.0 * c)) / 2.0, 0.0)

# ---- Ion channel gating ----------------------------------------------------

def f_p_open_KAT1_channel(Vmemb, V1by2_KAT1, delta_KAT1, hyd):
    """KAT1 inward-rectifying K⁺ channel open probability.
    
    Boltzmann gating: activates at hyperpolarized (negative) potentials.
    δ_KAT1 controls voltage sensitivity.
    """
    return 1.0 / (1.0 + np.exp((Vmemb - V1by2_KAT1)
                                * (delta_KAT1 * hyd["F"] / (hyd["R"] * hyd["T"]))))

def f_V1by2_GORK(Kout, hyd):
    """GORK outward-rectifying K⁺ channel half-activation voltage.
    
    Depends on external K⁺ concentration via a Nernst-like shift.
    """
    return 1e-3 * (hyd["F"] / (hyd["R"] * hyd["T"])) * np.log(Kout / hyd["V1by2_Kconc_GORK"])

def f_p_open_GORK_channel(Vmemb, V1by2_GORK, delta_GORK, hyd):
    """GORK outward-rectifying K⁺ channel open probability.
    
    Boltzmann gating: activates at depolarized (positive) potentials.
    """
    return 1.0 / (1.0 + np.exp(-(Vmemb - V1by2_GORK)
                                 * (delta_GORK * hyd["F"] / (hyd["R"] * hyd["T"]))))

# ---- Membrane transport (GHK fluxes, mass balance, H⁺-pump) ----------------

def f_GHK_current_flux(Pion, zion, Vmemb, ionConcIn, ionConcOut, hyd):
    """Goldman-Hodgkin-Katz current density for a single ion species.
    
    I = P · z² · F · ζ · (C_in - C_out·e^{-zζ}) / (1 - e^{-zζ})
    where ζ = V_m · F / (R·T).
    """
    zeta = Vmemb * hyd["F"] / (hyd["R"] * hyd["T"])
    return (Pion * zion**2 * hyd["F"] * zeta
            * (ionConcIn - ionConcOut * np.exp(-zion * zeta))
            / (1.0 - np.exp(-zion * zeta)))

def f_mass_balance(ionIn, ionIn0, ionOut0, hyd):
    """Apoplastic ion concentration from total ion conservation.
    
    Total moles = C_out0 · V_out + C_in0 · V_in = const.
    Given current intracellular [ion], returns current apoplastic [ion].
    """
    Q = ionOut0 * hyd["vol_out"] + ionIn0 * hyd["vol_in"]
    return (Q - ionIn * hyd["vol_in"]) / hyd["vol_out"]

def f_p_on_Hpump(ABA, hyd):
    """Fraction of H⁺-ATPase pumps active (ABA inhibits the pump).
    
    Hill inhibition: p_on = 1 / (1 + ([ABA]/K_half)^n)
    Rising ABA shuts down the proton pump → membrane depolarization.
    """
    return 1.0 / (1.0 + (ABA / hyd["ABA_Hpump_Khalf"])**hyd["ABA_Hpump_n"])

def f_HPump_current(p_on_Hpump, Hin, Hout, Vmemb, hyd):
    """H⁺-ATPase electrogenic pump current.
    
    Two-state kinetic model coupling ATP hydrolysis to proton extrusion.
    Net cycle rate = (k⁺₁·k⁺₂ - k⁻₁·k⁻₂) / (sum of all rate constants),
    scaled by pump density and p_on (ABA-dependent active fraction).
    """
    u       = Vmemb * hyd["F"] / (hyd["R"] * hyd["T"])
    kplus1  = hyd["k1pump"] * Hin
    kplus2  = hyd["k2pump"] * u / (1.0 - np.exp(-u))
    kminus1 = hyd["k1pump"] * np.exp(hyd["DeltaGATP"] / (hyd["R"] * hyd["T"]))
    kminus2 = hyd["k2pump"] * Hout * u * np.exp(-u) / (1.0 - np.exp(-u))
    cyc = (kplus1 * kplus2 - kminus1 * kminus2) / (kplus1 + kplus2 + kminus1 + kminus2)
    return p_on_Hpump * hyd["E0pump"] * cyc

def f_2HClSymport_current(p_on_Hpump, Hin, Hout, Clin, Clout, Vmemb, hyd):
    """2H⁺/Cl⁻ symporter current (secondary active Cl⁻ uptake).
    
    Driven by the proton gradient; transports 2 H⁺ + 1 Cl⁻ into the cell.
    """
    u = Vmemb * hyd["F"] / (hyd["R"] * hyd["T"])
    return p_on_Hpump * hyd["V0Cl"] * (
        Clin * Hin**2 * u - Clout * Hout**2 * u * np.exp(-u)) / (1.0 - np.exp(-u))

print("Helper functions defined.")


Helper functions defined.


## 4. ODE System

In [4]:
def fullModel_DeltaPg_ode(t, y, hyd):
    """
    Full 10-variable ODE system for the coupled HP-HA stomatal model.
    
    State vector y = [ψ_m, ψ_s, ψ_g, ABA, A8H, Oo, So, V_memb, Cl_in, K_in]
    
    Modules (in order of computation):
      1. Hydraulics      — tissue water potentials → turgor pressures → gs
      2. ABA biosynthesis — stress-sensing + autoregulatory motif → [ABA], [A8H]
      3. ABA signaling    — receptor–kinase–phosphatase cascade → OST1*, SLAC1*
      4. Electrophysiology — ion channels + pump → membrane potential + ion fluxes
    """
    # Unpack state variables
    psim, psis, psig, ABA, A8H, Oo, So, Vmemb, Clin, Kin = y

    # ---- Environmental forcing ----
    RH   = float(f_RH(t, hyd))
    psix = float(f_psix(t, hyd))
    pa   = hyd["psat"] * RH / 100.0            # ambient vapor pressure (MPa)

    # =========================================================================
    # MODULE 1: HYDRAULICS
    # Water flows through the SPAC: xylem → mesophyll → subsidiary → guard cell
    # with evaporative loss at the substomatal cavity
    # =========================================================================

    # Vapor pressures at each tissue surface (Clausius-Clapeyron linearization)
    pm = hyd["psat"] * (1.0 + psim / hyd["RT_by_nu_by1e6"])
    ps = hyd["psat"] * (1.0 + psis / hyd["RT_by_nu_by1e6"])
    pg = hyd["psat"] * (1.0 + psig / hyd["RT_by_nu_by1e6"])

    # Gas-phase resistances from guard cell and epidermis to substomatal cavity
    Rg   = (1.0 / hyd["gg1"]) * hyd["RT_by_nu_by1e6"] * hyd["patm"] / hyd["psat"]
    Re   = (1.0 / hyd["ge1"]) * hyd["RT_by_nu_by1e6"] * hyd["patm"] / hyd["psat"]
    Req1 = Re / (1.0 + Re / (hyd["Rsg"] + Rg))

    # Overall xylem resistance (vulnerability-curve dependent)
    Rox = f_Rox(psix)
    Rm  = (Rox - hyd["Rxm"]) / (1.0 + (Rox - hyd["Rxm"]) / (hyd["Rms"] + Req1))

    # Mesophyll-to-SSC conductance (mol H₂O / m² / s / MPa)
    gmH2O = (1.0 / Rm) * hyd["RT_by_nu_by1e6"] * hyd["patm"] / hyd["psat"]

    # Guard cell osmotic pressure change from ion fluxes (MPa)
    DeltaPig_MPa = (hyd["Kin0"] + hyd["Clin0"] - Kin - Clin + hyd["otherIons"]) \
                   * hyd["R"] * hyd["T"] / 1e6

    # Turgor pressures (MPa)
    Pg = f_P_from_psi(psig, hyd["epsilong"], hyd["pig0"], hyd["Pg0"], DeltaPig_MPa)
    Ps = f_P_from_psi(psis, hyd["epsilons"], hyd["pis0"], hyd["Ps0"], 0.0)

    # Stomatal conductance (mmol/m²/s)
    gs = max(hyd["Xi"] * (Pg - hyd["m"] * Ps), 0.0)

    # Substomatal cavity water potential (SSC node in hydraulic circuit)
    psii = f_SSC_apoplast_potential(psim, psig, psis, gs, gmH2O, pa, hyd,
                                    hyd["ge1"], hyd["gg1"])
    p_i  = hyd["psat"] * (1.0 + psii / hyd["RT_by_nu_by1e6"])

    # Evaporative fluxes from each tissue to the SSC and atmosphere
    Em  = gmH2O       * (pm - p_i) / hyd["patm"]   # mesophyll → SSC
    Es1 = hyd["ge1"]  * (ps - p_i) / hyd["patm"]   # subsidiary → SSC
    Eg1 = hyd["gg1"]  * (pg - p_i) / hyd["patm"]   # guard → SSC
    Es2 = hyd["ge2"]  * (ps - pa)  / hyd["patm"]    # subsidiary → atmosphere (cuticular)
    Eg2 = hyd["gg2"]  * (pg - pa)  / hyd["patm"]    # guard → atmosphere (cuticular)

    # Tissue hydraulic capacitances: C = V / (ε + π₀)
    Cm_c = hyd["Vm0"] / (hyd["epsilonm"] + hyd["pim0"])
    Cs_c = hyd["Vs0"] / (hyd["epsilons"] + hyd["pis0"])
    Cg_c = hyd["Vg0"] / (hyd["epsilong"] + hyd["pig0"])

    # dψ/dt for the three tissue compartments (Kirchhoff's current law)
    dpsimdt = (1 / Cm_c) * (psix / hyd["Rxm"]
              + psim * (-1 / hyd["Rxm"] - 1 / hyd["Rms"])
              + psis / hyd["Rms"] - Em)

    dpsisdt = (1 / Cs_c) * (psim / hyd["Rms"]
              + psis * (-1 / hyd["Rms"] - 1 / hyd["Rsg"])
              + psig / hyd["Rsg"] - Es1 - Es2)

    dpsigdt = (1 / Cg_c) * (psis / hyd["Rsg"]
              + psig * (-1 / hyd["Rsg"]) - Eg1 - Eg2)

    # =========================================================================
    # MODULE 2: ABA BIOSYNTHESIS & CATABOLISM
    # Autoregulatory motif: stress-driven NCED (+ feedback) vs CYP707A/A8H
    # =========================================================================

    # Turgor loss drives stress-responsive ABA synthesis (Hill function)
    DeltaPg_val = Pg - hyd["Pg0"]
    Vstress_val = f_Vstress(DeltaPg_val, hyd)

    # d[ABA]/dt = synthesis (stress + positive feedback) − catabolism by A8H
    dABAdt = (hyd["thetaABA"]
              * (Vstress_val + hyd["Vplus"] * ABA**hyd["n"]
                 / (hyd["K1"]**hyd["n"] + ABA**hyd["n"]))
              - hyd["kcat"] * A8H * ABA / hyd["K2"])

    # d[A8H]/dt = ABA-induced A8H production − protein decay
    dA8Hdt = (hyd["thetaA8H"] * hyd["Vminus"] * ABA / (hyd["K3"] + ABA)
              - hyd["gammac"] * A8H)

    # =========================================================================
    # MODULE 3: ABA SIGNALING CASCADE
    # Bicyclic futile cycle: MAP3K → OST1 → SLAC1, with PP2C dephosphorylation
    # inhibited by ABA:PYR complex (non-competitive)
    # =========================================================================

    # Non-dimensionalized Michaelis constants (by total protein concentrations)
    K1_nd = hyd["Km1"] / hyd["OST1T"]
    K2_nd = hyd["Km2"] / hyd["OST1T"]
    K3_nd = hyd["Km3"] / hyd["SLAC1T"]
    K4_nd = hyd["Km4"] / hyd["SLAC1T"]

    # Maximal rates
    V1 = hyd["k1"] * hyd["MAP3KT"]     # MAP3K → OST1 phosphorylation
    V2 = hyd["k2"] * hyd["PP2CT"]      # PP2C ⊣ OST1 dephosphorylation
    V3 = hyd["k3"] * hyd["OST1T"]      # OST1 → SLAC1 phosphorylation
    V4 = hyd["k4"] * hyd["PP2CT"]      # PP2C ⊣ SLAC1 dephosphorylation

    epsilon1 = hyd["MAP3KT"] / hyd["OST1T"]

    # Bundle non-dimensional constants for conservation law solvers
    hx = {**hyd, "K1_nd": K1_nd, "K2_nd": K2_nd,
           "K3_nd": K3_nd, "K4_nd": K4_nd, "epsilon1": epsilon1}

    # ABA:PYR receptor binding → effective PP2C inhibition
    ABA_PYR = f_solve_ABA_PYR_quadratic(ABA, hyd)
    V2p = V2 / (1.0 + ABA_PYR / hyd["KI"])    # reduced PP2C activity on OST1
    V4p = V4 / (1.0 + ABA_PYR / hyd["KI"])    # reduced PP2C activity on SLAC1

    # Apparent Michaelis constants (cross-competition between the two cycles)
    K2prime = K2_nd * (1.0 + So / K4_nd)
    K4prime = K4_nd * (1.0 + Oo / K2_nd)

    # dOo/dt — OST1 phosphorylation dynamics (Oo = OST1* / OST1_total)
    O = f_OST1_roots_from_quadratic_non_dimensional(Oo, So, hx)
    dOodt = (V2p / hyd["OST1T"]) * ((V1 / V2p) * O / (K1_nd + O)
             - Oo / (K2prime + Oo))

    # dSo/dt — SLAC1 phosphorylation dynamics (So = SLAC1* / SLAC1_total)
    S = f_SLAC1_conc_non_dimensional(Oo, So, hx)
    dSodt = (V4p / hyd["SLAC1T"]) * ((V3 / V4p) * Oo * S / K3_nd
             - So / (K4prime + So))

    # =========================================================================
    # MODULE 4: GUARD CELL ELECTROPHYSIOLOGY
    # Ion channels (SLAC1, KAT1, GORK) + H⁺-ATPase → V_memb, Cl⁻, K⁺ dynamics
    # =========================================================================

    # Apoplastic ion concentrations (from mass balance)
    Clout = f_mass_balance(Clin, hyd["Clin0"], hyd["Clout0"], hyd)
    Kout  = f_mass_balance(Kin,  hyd["Kin0"],  hyd["Kout0"],  hyd)

    # --- SLAC1 Cl⁻ efflux (activated by SLAC1* phosphorylation) ---
    IGHK_Cl   = f_GHK_current_flux(hyd["PCl"], hyd["zCl"], Vmemb, Clin, Clout, hyd)
    ICl_SLAC1 = (hyd["SLAC1T"] / 1e3) * So * IGHK_Cl

    # --- KAT1 K⁺ influx (inward rectifier, voltage-gated) ---
    p_KAT1  = f_p_open_KAT1_channel(Vmemb, hyd["V1by2_KAT1"], hyd["delta_KAT1"], hyd)
    IGHK_K  = f_GHK_current_flux(hyd["PK"], hyd["zK"], Vmemb, Kin, Kout, hyd)
    IK_KAT1 = hyd["NKAT1"] * p_KAT1 * IGHK_K

    # --- GORK K⁺ efflux (outward rectifier, voltage-gated) ---
    V12G    = f_V1by2_GORK(Kout, hyd)
    p_GORK  = f_p_open_GORK_channel(Vmemb, V12G, hyd["delta_GORK"], hyd)
    IK_GORK = hyd["NGORK"] * p_GORK * IGHK_K

    # --- H⁺-ATPase proton pump (ABA inhibits → depolarization) ---
    p_pump  = f_p_on_Hpump(ABA, hyd)
    IH_pump = f_HPump_current(p_pump, hyd["Hin0"], hyd["Hout0"], Vmemb, hyd)

    # --- 2H⁺/Cl⁻ symporter (secondary active Cl⁻ uptake) ---
    ICl_sym = f_2HClSymport_current(1.0, hyd["Hin0"], hyd["Hout0"],
                                     Clin, Clout, Vmemb, hyd)

    # dV_memb/dt — membrane potential (sum of all ionic currents / capacitance)
    dVmembdt = -(1 / hyd["Cm"]) * (IH_pump + ICl_SLAC1 + IK_KAT1 + IK_GORK + ICl_sym)

    # dCl_in/dt and dK_in/dt — intracellular ion concentration changes
    dClindt = -(ICl_SLAC1 - ICl_sym) * hyd["zCl"] * hyd["area"] / (hyd["F"] * hyd["vol_in"])
    dKindt  = -(IK_GORK + IK_KAT1)  * hyd["zK"]  * hyd["area"] / (hyd["F"] * hyd["vol_in"])

    return [dpsimdt, dpsisdt, dpsigdt, dABAdt, dA8Hdt,
            dOodt, dSodt, dVmembdt, dClindt, dKindt]


def evaluate_output(t, y, hyd):
    """Post-process ODE solution into physically meaningful quantities.
    
    Takes raw solver output (time vector t, state matrix y) and computes
    derived variables: turgor pressures, stomatal conductance, RH forcing.
    Also creates a shifted time axis (t_plt) centered on the VPD step.
    """
    # Unpack state trajectories
    psim  = y[:, 0]; psis = y[:, 1]; psig = y[:, 2]
    ABA   = y[:, 3]; A8H  = y[:, 4]
    Oo    = y[:, 5]; So   = y[:, 6]
    Vmemb = y[:, 7]; Clin = y[:, 8]; Kin = y[:, 9]

    # Time axis relative to VPD step (for plotting)
    t_plt = t - hyd["tstep1"]

    # Recompute turgor and gs from state variables
    DeltaPig_MPa = (hyd["Kin0"] + hyd["Clin0"] - Kin - Clin + hyd["otherIons"]) \
                   * hyd["R"] * hyd["T"] / 1e6
    Pg = f_P_from_psi(psig, hyd["epsilong"], hyd["pig0"], hyd["Pg0"], DeltaPig_MPa)
    Ps = f_P_from_psi(psis, hyd["epsilons"], hyd["pis0"], hyd["Ps0"], 0.0)

    # Stomatal conductance (mmol/m²/s)
    gs = np.maximum(hyd["Xi"] * (Pg - hyd["m"] * Ps), 0.0)

    # Relative gs (normalized to value at VPD step)
    RH = f_RH(t, hyd).astype(float)
    tstep_idx = np.argmin(np.abs(t_plt))
    gs_relative = gs / max(gs[tstep_idx], 1e-12)

    # Additional derived quantities for diagnostic plots
    psix = np.full_like(t, hyd["psix_at_RH_step"])
    VPD  = f_VPD(t, hyd).astype(float)
    DeltaPg = Pg - hyd["Pg0"]
    Vstress = hyd["thetaABA"] * f_Vstress(DeltaPg, hyd)

    return dict(t=t, t_plt=t_plt, ABA=ABA, A8H=A8H, gs=gs, gs_relative=gs_relative,
                RH=RH, Pg=Pg, Ps=Ps, Oo=Oo, So=So, Vmemb=Vmemb, Clin=Clin, Kin=Kin,
                psix=psix, VPD=VPD, Vstress=Vstress, DeltaPig_MPa=DeltaPig_MPa)


def run_fullModel(hyd):
    """Integrate the full 10-variable ODE system for Figure 2.
    
    Initial conditions: all water potential perturbations = 0, [ABA] ≈ 0,
    [A8H] ≈ 0, no signaling activity, resting membrane potential = −180 mV,
    ion concentrations at baseline.
    
    Uses BDF (implicit) solver appropriate for this stiff system.
    Output sampled every 30 seconds.
    """
    tspan = np.arange(hyd["tstart"], hyd["tend"] + 30, 30.0)

    y0 = [0.0,             # ψ_m  — mesophyll water potential perturbation
          0.0,             # ψ_s  — subsidiary water potential perturbation
          0.0,             # ψ_g  — guard cell water potential perturbation
          1e-3,            # [ABA] (nM) — trace initial
          1e-3,            # [A8H] (nM) — trace initial
          0.0,             # Oo — OST1* fraction (inactive)
          0.0,             # So — SLAC1* fraction (inactive)
          -180e-3,         # V_memb (V) — resting membrane potential
          hyd["Clin0"],    # [Cl⁻]_in (mM)
          hyd["Kin0"]]     # [K⁺]_in  (mM)

    sol = solve_ivp(
        lambda t, y: fullModel_DeltaPg_ode(t, y, hyd),
        [hyd["tstart"], hyd["tend"]],
        y0, method="BDF", t_eval=tspan, rtol=1e-6, atol=1e-9)

    return evaluate_output(sol.t, sol.y.T, hyd)

print("ODE and evaluation functions defined.")


ODE and evaluation functions defined.


## 5. Simulation Engine

In [5]:
def run_figure2(hyd_overrides=None, progress_label=None):
    """Run Figure 2 VPD step simulation with parameter overrides.
    
    Supports direct overrides of Vstress, Vplus, Vminus (from ABA Biosynthesis
    dashboard) — these take priority over phi-derived values.
    Also supports K_pm (combined K+=K- slider) that sets both K1 and K3.
    """
    hyd = read_parameters_wt_Merilo_2018()
    if hyd_overrides:
        # Check if direct ABA rates were provided
        direct_rates = {}
        for k in ("Vstress", "Vplus", "Vminus"):
            if k in hyd_overrides:
                direct_rates[k] = hyd_overrides.pop(k)

        # Check for combined K+=K- override
        K_pm_val = hyd_overrides.pop("K_pm", None)

        for k, v in hyd_overrides.items():
            hyd[k] = v

        # Recompute derived quantities
        hyd["K1"] = hyd["lambda1"] * hyd["K2"]
        hyd["K3"] = hyd["lambda3"] * hyd["K2"]
        hyd["pig0"] = hyd["Pg0"] + 0.1
        hyd["pis0"] = hyd["Ps0"] + 0.1
        hyd["pim0"] = hyd["Pm0"] + 0.1
        hyd["vol_in"] = hyd["Vg0"] * hyd["nu_w"] * 1e-3 / hyd["stomatal_density"]
        hyd["VPD1"] = hyd["psat"] * (1 - hyd["RH1"]/100)
        hyd["VPD2"] = hyd["psat"] * (1 - hyd["RH2"]/100)

        # Apply combined K+=K- (overrides lambda-derived K1, K3)
        if K_pm_val is not None:
            hyd["K1"] = K_pm_val
            hyd["K3"] = K_pm_val

        if direct_rates:
            # Direct ABA rates override phi-derived values
            hyd["Vstress"] = direct_rates.get("Vstress", hyd["Vstress"])
            hyd["Vplus"]   = direct_rates.get("Vplus",   hyd["Vplus"])
            hyd["Vminus"]  = direct_rates.get("Vminus",  hyd["Vminus"])
        else:
            # Recompute from phi values (legacy path for other dashboards)
            hyd["Vminus"] = (hyd["K2"] * hyd["gammac"]**2 * hyd["phi3"]) / hyd["kcat"]
            hyd["Vplus"]  = hyd["phi2"] * hyd["kcat"] * hyd["Vminus"] / hyd["gammac"]
            hyd["Vstress"]= hyd["phi1"] * hyd["kcat"] * hyd["Vminus"] / hyd["gammac"]

    if progress_label:
        progress_label.value = 'Integrating ODE...'
    result = run_fullModel(hyd)
    if progress_label:
        progress_label.value = 'Done!'
    return result

print('Figure 2 simulation engine ready.')


Figure 2 simulation engine ready.


## 6. Run All Three Genotypes

Wild-type (Col-0), signaling mutant (*ost1-3*), and synthesis mutant (*aao3-2*).

In [6]:
# ── Wild-type baseline (used by dashboards) ──
print("Running wild-type (Col-0) ...")
baseline = run_figure2()
hyd_out_wt = baseline
print("  done.")

# ── Signaling mutant: ost1-3 (OST1 near zero) ──
print("Running signaling mutant (ost1-3) ...")
hyd_out_sig = run_figure2({"OST1T": 1e-3})
print("  done.")

# ── Synthesis mutant: aao3-2 (NCED knocked out) ──
print("Running synthesis mutant (aao3-2) ...")
hyd_out_syn = run_figure2({"thetaABA": 0.12, "n": 2})
print("  done.")

xlim_L, xlim_R = -20, 60


Running wild-type (Col-0) ...
  done.
Running signaling mutant (ost1-3) ...
  done.
Running synthesis mutant (aao3-2) ...
  done.


### Load Merilo 2018 experimental data

In [ ]:
# Robust CSV loader: local file when available (e.g. running locally or during
# the panel convert build step), raw GitHub URL when running inside Pyodide
# in the browser, where there is no filesystem.
import os

DATA_BASE = (
    "https://raw.githubusercontent.com/"
    "desai-sahil/stomatal-conductance-model/main/data/"
)

def _load_csv(filename):
    local = os.path.join("data", filename)
    src = local if os.path.exists(local) else (DATA_BASE + filename)
    return pd.read_csv(src, skipinitialspace=True)

m_wt    = _load_csv("merilo_2018_wildtype_vpd_change.csv")
m_ost1  = _load_csv("merilo_2018_ost1_vpd_change.csv")
m_aao32 = _load_csv("merilo_2018_aao32_vpd_change.csv")

m_wt["yerr"]    = 14.0
m_ost1["yerr"]  = 32.0
m_aao32["yerr"] = 26.0

print(f"Loaded experimental data: WT ({len(m_wt)} pts), ost1-3 ({len(m_ost1)} pts), aao3-2 ({len(m_aao32)} pts)")


## 7. Figure 2: Model vs Experiment

Comparison of model predictions against Merilo 2018 experimental data for all three genotypes.

In [ ]:
fig_transients_1, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Absolute gs ──
ax = axes[0]
ax.errorbar(m_wt["x"],    m_wt["y"],    m_wt["yerr"],    fmt="g-o", mfc="g", ms=8, label="Col-0")
ax.errorbar(m_ost1["x"],  m_ost1["y"],  m_ost1["yerr"],  fmt="b-^", mfc="b", ms=8, label="ost1-3")
ax.errorbar(m_aao32["x"], m_aao32["y"], m_aao32["yerr"], fmt="r-d", mfc="r", ms=8, label="aao3-2")
ax.plot(hyd_out_wt["t_plt"]/60,  hyd_out_wt["gs"],  "-g", lw=2)
ax.plot(hyd_out_sig["t_plt"]/60, hyd_out_sig["gs"], "-b", lw=2)
ax.plot(hyd_out_syn["t_plt"]/60, hyd_out_syn["gs"], "-r", lw=2)
ax.set_xlim(xlim_L, xlim_R); ax.set_ylim(0, 400)
ax.set_xlabel("Time (min)"); ax.set_ylabel("gs (mmol/m\u00b2/s)"); ax.set_title("All genotypes — Absolute gs")
ax.minorticks_on(); ax.grid(True, which="both", alpha=0.3); ax.legend()

# ── Relative gs ──
ax = axes[1]
ax.errorbar(m_wt["x"],    m_wt["y"]/m_wt["y"].iloc[7],
            m_wt["yerr"]/m_wt["y"].iloc[7],       fmt="g-o", mfc="g", ms=8, label="Col-0")
ax.errorbar(m_ost1["x"],  m_ost1["y"]/m_ost1["y"].iloc[6],
            m_ost1["yerr"]/m_ost1["y"].iloc[6],   fmt="b-o", mfc="b", ms=8, label="ost1-3")
ax.errorbar(m_aao32["x"], m_aao32["y"]/m_aao32["y"].iloc[6],
            m_aao32["yerr"]/m_aao32["y"].iloc[6], fmt="r-o", mfc="r", ms=8, label="aao3-2")
ax.plot(hyd_out_wt["t_plt"]/60,  hyd_out_wt["gs_relative"],  "-g", lw=2)
ax.plot(hyd_out_sig["t_plt"]/60, hyd_out_sig["gs_relative"], "-b", lw=2)
ax.plot(hyd_out_syn["t_plt"]/60, hyd_out_syn["gs_relative"], "-r", lw=2)
ax.set_xlim(xlim_L, xlim_R)
ax.set_xlabel("Time (min)"); ax.set_ylabel("Relative gs"); ax.set_title("All genotypes — Relative gs")
ax.minorticks_on(); ax.grid(True, which="both", alpha=0.3); ax.legend()

plt.tight_layout(); plt.show()


### Physiological State Variables

In [ ]:
fig_transients_2, axes = plt.subplots(3, 3, figsize=(20, 14))
t = hyd_out_wt["t_plt"] / 60

# ── Row 1: Environmental forcing ──────────────────────────────────────────────

axes[0,0].plot(t, hyd_out_wt["psix"], "-k", lw=2)
axes[0,0].set_xlim(xlim_L, xlim_R); axes[0,0].set_ylim(-1.5, 0)
axes[0,0].set_xlabel("Time (min)"); axes[0,0].set_ylabel("\u03c8\u2093 (MPa)")
axes[0,0].set_title("Xylem water potential"); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(t, hyd_out_wt["VPD"]*1e3, "-k", lw=2)
axes[0,1].set_xlim(xlim_L, xlim_R); axes[0,1].set_ylim(0, 4)
axes[0,1].set_xlabel("Time (min)"); axes[0,1].set_ylabel("VPD (kPa)")
axes[0,1].set_title("Vapour pressure deficit"); axes[0,1].grid(alpha=0.3)

axes[0,2].plot(t, hyd_out_wt["RH"], "-k", lw=2)
axes[0,2].set_xlim(xlim_L, xlim_R); axes[0,2].set_ylim(50, 100)
axes[0,2].set_xlabel("Time (min)"); axes[0,2].set_ylabel("RH (%)")
axes[0,2].set_title("Relative humidity step"); axes[0,2].grid(alpha=0.3)

# ── Row 2: Stomatal response ─────────────────────────────────────────────────

for o, col, lbl in [(hyd_out_wt, "-g", "WT"), (hyd_out_syn, "-r", "aao3-2"), (hyd_out_sig, "-b", "ost1-3")]:
    axes[1,0].plot(o["t_plt"]/60, o["gs"], col, lw=2, label=lbl)
axes[1,0].set_xlim(xlim_L, xlim_R); axes[1,0].set_ylim(0, 400)
axes[1,0].set_xlabel("Time (min)"); axes[1,0].set_ylabel("gs (mmol/m\u00b2/s)")
axes[1,0].set_title("Stomatal conductance"); axes[1,0].grid(alpha=0.3); axes[1,0].legend(fontsize=10)

for o, col, lbl in [(hyd_out_wt, "-g", "WT"), (hyd_out_syn, "-r", "aao3-2"), (hyd_out_sig, "-b", "ost1-3")]:
    axes[1,1].plot(o["t_plt"]/60, o["Pg"], col, lw=2, label=lbl)
axes[1,1].set_xlim(xlim_L, xlim_R)
axes[1,1].set_xlabel("Time (min)"); axes[1,1].set_ylabel("P_g (MPa)")
axes[1,1].set_title("Guard cell turgor"); axes[1,1].grid(alpha=0.3); axes[1,1].legend(fontsize=10)

for o, col, lbl in [(hyd_out_wt, "-g", "WT"), (hyd_out_syn, "-r", "aao3-2"), (hyd_out_sig, "-b", "ost1-3")]:
    axes[1,2].plot(o["t_plt"]/60, o["Vstress"], col, lw=2, label=lbl)
axes[1,2].set_xlim(xlim_L, xlim_R)
axes[1,2].set_xlabel("Time (min)"); axes[1,2].set_ylabel("V_stress (nM/s)")
axes[1,2].set_title("Stress-induced ABA synthesis"); axes[1,2].grid(alpha=0.3); axes[1,2].legend(fontsize=10)

# ── Row 3: ABA signaling ─────────────────────────────────────────────────────

for o, col, lbl in [(hyd_out_wt, "-g", "WT"), (hyd_out_syn, "-r", "aao3-2"), (hyd_out_sig, "-b", "ost1-3")]:
    axes[2,0].plot(o["t_plt"]/60, o["ABA"], col, lw=2, label=lbl)
axes[2,0].set_xlim(xlim_L, xlim_R); axes[2,0].set_ylim(0, 1000)
axes[2,0].set_xlabel("Time (min)"); axes[2,0].set_ylabel("[ABA] (nM)")
axes[2,0].set_title("ABA concentration"); axes[2,0].grid(alpha=0.3); axes[2,0].legend(fontsize=10)

for o, col, lbl in [(hyd_out_wt, "-g", "WT"), (hyd_out_syn, "-r", "aao3-2"), (hyd_out_sig, "-b", "ost1-3")]:
    axes[2,1].plot(o["t_plt"]/60, o["So"], col, lw=2, label=lbl)
axes[2,1].set_xlim(xlim_L, xlim_R); axes[2,1].set_ylim(0, 0.8)
axes[2,1].set_xlabel("Time (min)"); axes[2,1].set_ylabel("S_o (\u2014)")
axes[2,1].set_title("Active SLAC1 fraction"); axes[2,1].grid(alpha=0.3); axes[2,1].legend(fontsize=10)

for o, col, lbl in [(hyd_out_wt, "-g", "WT"), (hyd_out_syn, "-r", "aao3-2"), (hyd_out_sig, "-b", "ost1-3")]:
    axes[2,2].plot(o["t_plt"]/60, -o["DeltaPig_MPa"], col, lw=2, label=lbl)
axes[2,2].set_xlim(xlim_L, xlim_R)
axes[2,2].set_xlabel("Time (min)"); axes[2,2].set_ylabel("\u0394\u03c0_g (MPa)")
axes[2,2].set_title("Guard cell osmotic pressure change"); axes[2,2].grid(alpha=0.3); axes[2,2].legend(fontsize=10)

plt.tight_layout(); plt.show()


### Guard Cell Electrophysiology (WT)

In [ ]:
fig_transients_3, axes = plt.subplots(1, 3, figsize=(20, 5))
t = hyd_out_wt["t_plt"] / 60

axes[0].plot(t, hyd_out_wt["Vmemb"]*1e3, "-k", lw=2)
axes[0].set_xlim(xlim_L, xlim_R)
axes[0].set_xlabel("Time (min)"); axes[0].set_ylabel("V_memb (mV)")
axes[0].set_title("Guard cell membrane potential (WT)")
axes[0].minorticks_on(); axes[0].grid(True, which="both", alpha=0.3)

axes[1].plot(t, hyd_out_wt["Clin"], "-r", lw=2)
axes[1].set_xlim(xlim_L, xlim_R)
axes[1].set_xlabel("Time (min)"); axes[1].set_ylabel("[Cl\u207b]_in (mM)")
axes[1].set_title("Guard cell Cl\u207b (WT)")
axes[1].minorticks_on(); axes[1].grid(True, which="both", alpha=0.3)

axes[2].plot(t, hyd_out_wt["Kin"], "-b", lw=2)
axes[2].set_xlim(xlim_L, xlim_R)
axes[2].set_xlabel("Time (min)"); axes[2].set_ylabel("[K\u207a]_in (mM)")
axes[2].set_title("Guard cell K\u207a (WT)")
axes[2].minorticks_on(); axes[2].grid(True, which="both", alpha=0.3)

plt.tight_layout(); plt.show()


## 8. Dashboard Infrastructure

In [11]:
from bokeh.plotting import figure
from bokeh.layouts import column as bokeh_column, row as bokeh_row

def make_figure2_plot(result, title_suffix):
    t_min = result["t_plt"] / 60
    t_base = baseline["t_plt"] / 60

    p1 = figure(title=f'[ABA] — {title_suffix}', x_axis_label='Time (min)', y_axis_label='[ABA] (nM)',
                width=450, height=300, x_range=(-20, 60))
    p1.line(list(t_base), list(baseline["ABA"]), line_width=1.5, color='gray', alpha=0.5, legend_label='Baseline')
    p1.line(list(t_min), list(result["ABA"]), line_width=2.5, color='red', legend_label='Modified')
    p1.legend.location = 'top_right'

    p2 = figure(title=f'gs — {title_suffix}', x_axis_label='Time (min)', y_axis_label='gs (mmol/m²/s)',
                width=450, height=300, x_range=(-20, 60))
    p2.line(list(t_base), list(baseline["gs"]), line_width=1.5, color='gray', alpha=0.5, legend_label='Baseline')
    p2.line(list(t_min), list(result["gs"]), line_width=2.5, color='blue', legend_label='Modified')
    p2.legend.location = 'top_right'

    return bokeh_row(p1, p2)

def make_dashboard(title, description, slider_specs, dashboard_name):
    """Create a parameter-exploration dashboard.
    
    slider_specs: list of tuples. Each tuple is either:
        (key, label, lo, hi, default, step)              — no unit conversion
        (key, label, lo, hi, default, step, scale_factor) — slider values are DIVIDED
            by scale_factor before passing to the ODE.
            Example: P_K = 5e-6 m/s displayed as 5.0 µm/s with scale_factor=1e6.
    """
    sliders = {}
    slider_widgets = []
    scale_factors = {}
    for spec in slider_specs:
        if len(spec) == 7:
            key, label, lo, hi, default, step_sz, sf = spec
            scale_factors[key] = sf
        else:
            key, label, lo, hi, default, step_sz = spec
            scale_factors[key] = 1.0
        s = pn.widgets.FloatSlider(name=label, start=lo, end=hi, value=default, step=step_sz, width=400)
        sliders[key] = s
        slider_widgets.append(s)

    status = pn.widgets.StaticText(value='Ready. Adjust sliders then click Compute.')
    compute_btn = pn.widgets.Button(name='Compute', button_type='success', width=200)
    reset_btn = pn.widgets.Button(name='Reset Defaults', button_type='warning', width=200)

    plot_pane = pn.pane.Bokeh(make_figure2_plot(baseline, f'{dashboard_name} (baseline)'))

    def on_compute(event):
        status.value = 'Computing... (this may take a few minutes)'
        overrides = {}
        for k, s in sliders.items():
            overrides[k] = s.value / scale_factors[k]
        try:
            result = run_figure2(overrides)
            plot_pane.object = make_figure2_plot(result, dashboard_name)
            status.value = 'Done!'
        except Exception as e:
            status.value = f'Error: {e}'

    def on_reset(event):
        for spec in slider_specs:
            key, default = spec[0], spec[4]
            sliders[key].value = default
        status.value = 'Sliders reset to defaults.'

    compute_btn.on_click(on_compute)
    reset_btn.on_click(on_reset)

    n = len(slider_widgets)
    mid = (n + 1) // 2
    col1 = pn.Column(*slider_widgets[:mid])
    col2 = pn.Column(*slider_widgets[mid:])
    slider_row = pn.Row(col1, col2)

    header = pn.pane.Markdown(f'### {title}\n{description}')
    buttons = pn.Row(compute_btn, reset_btn)

    return pn.Column(header, slider_row, buttons, status, plot_pane)

print('Dashboard infrastructure ready (with unit-scaling support).')

Dashboard infrastructure ready (with unit-scaling support).


## Dashboard Usage Tips

**How to use these dashboards:**

1. **Adjust sliders** — Each slider controls one parameter from the corresponding group.
2. **Click Compute** — This re-runs the entire 240,000 second VPD step simulation with your new parameters (takes ~1–3 minutes).
3. **Compare curves** — The modified run (colored lines) is plotted alongside the baseline (gray).
4. **Reset Defaults** — Return all sliders to their defaults and re-initialize the plot.

**Tips for exploration:**

- **Small changes first** — Try ±10% adjustments to see sensitivity.
- **Isolate effects** — Change one dashboard at a time to understand each parameter group.

**Interpretation:**

- **RH curve** shows the environmental step (instantaneous, 80% → 60%).
- **ABA curve** shows how quickly the stress hormone responds and recovers.
- **gs (relative)** shows fractional stomatal conductance (normalized to baseline at t = 0).

For details, see **Merilo et al. (2018)** in *New Phytologist*.

---
## Dashboard 1: Hydraulics

These parameters control **water transport** from xylem through mesophyll to guard cells, and the force balance that converts turgor to stomatal opening.

In [12]:
dashboard_hydraulics = make_dashboard(
    title='Hydraulics Parameters',
    description='Adjust tissue resistances and stomatal conductance equation coefficients.',
    slider_specs=[
        # (key,    label,                      min,   max,   default, step)
        ('Rms',   'R_ms (MPa/mmol/m²/s)',         0.5,  20.0,   5.0,    0.5),
        ('Rsg',   'R_sg (MPa/mmol/m²/s)',         1.0,  30.0,  10.0,    0.5),
        ('gg1',   'g_g¹ (mmol/m²/s)',     0.5,  10.0,   2.9,    0.1),
        ('gg2',   'g_g² (mmol/m²/s)',            0.2,   3.0,   1.0,    0.1),
        ('ge1',   'g_e¹ (mmol/m²/s)',     0.5,  10.0,   2.9,    0.1),
        ('ge2',   'g_e² (mmol/m²/s)',            1.0,  30.0,  10.0,    0.5),
    ],
    dashboard_name='Hydraulics'
)
dashboard_hydraulics


Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=20.0, name='R_ms (MPa/mmol/m²/s)', start=0.5, step=0.5, value=5.0, width=400)
            [1] FloatSlider(end=30.0, name='R_sg (MPa/mmol/m²/s)', start=1.0, step=0.5, value=10.0, width=400)
            [2] FloatSlider(end=10.0, name='g_g¹ (mmol/m²/s)', start=0.5, value=2.9, width=400)
        [1] Column
            [0] FloatSlider(end=3.0, name='g_g² (mmol/m²/s)', start=0.2, value=1.0, width=400)
            [1] FloatSlider(end=10.0, name='g_e¹ (mmol/m²/s)', start=0.5, value=2.9, width=400)
            [2] FloatSlider(end=30.0, name='g_e² (mmol/m²/s)', start=1.0, step=0.5, value=10.0, width=400)
    [2] Row
        [0] Button(button_type='success', name='Compute', width=200)
        [1] Button(button_type='warning', name='Reset Defaults', width=200)
    [3] StaticText(value='Ready. Adjust s...)
    [4] Bokeh(Row)

---
## Dashboard 2: Stress Sensing (Vstress → ΔPg Curve)

These parameters shape the **stress-responsive ABA synthesis pathway**, modulating how guard cell turgor loss triggers ABA production.

In [13]:
dashboard_vstress = make_dashboard(
    title='Vstress → ΔPg Curve',
    description='How guard cell turgor loss triggers stress-responsive ABA synthesis.',
    slider_specs=[
        ('vstress_exponent',  'Hill exponent n',     1.0,  6.0,  3.25,  0.25),
        ('vstress_khalf',     'K_half (MPaⁿ)',       0.2,  5.0,  1.4,   0.1),
    ],
    dashboard_name='Vstress Curve'
)
dashboard_vstress


Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=6.0, name='Hill exponent n', start=1.0, step=0.25, value=3.25, width=400)
        [1] Column
            [0] FloatSlider(end=5.0, name='K_half (MPaⁿ)', start=0.2, value=1.4, width=400)
    [2] Row
        [0] Button(button_type='success', name='Compute', width=200)
        [1] Button(button_type='warning', name='Reset Defaults', width=200)
    [3] StaticText(value='Ready. Adjust s...)
    [4] Bokeh(Row)

---
## Dashboard 3: ABA Biosynthesis & Catabolism

These parameters control the **production and breakdown of ABA** via the autoregulation motif (SI Section S3.1, Eqs. S5, S8): stress-induced synthesis ($V_{stress}$), positive feedback ($V_+$, $K_+$), enzymatic degradation by A8H ($V_-$, $K_d$, $\gamma_A$), and A8H turnover ($K_-$, $\gamma_H$).

In [14]:
dashboard_aba = make_dashboard(
    title='ABA Biosynthesis & Catabolism',
    description='Synthesis rates, feedback gains, and enzyme kinetics of the ABA autoregulatory motif.',
    slider_specs=[
        # key        label                              min     max     default   step   [scale]
        ('Vstress', 'V_stress (nM/s)',                   1.0,   30.0,    8.85,   0.1),
        ('Vplus',   'V₊ (nM/s)',                        10.0,  200.0,   67.58,   1.0),
        ('Vminus',  'V₋ (nM/s)',                         0.1,   10.0,    1.51,   0.05),
        ('K_pm',    'K₊ = K₋ (nM)',                    200.0, 5000.0, 1560.0,  20.0),
        ('K2',      'K_d (nM)',                        500.0, 5000.0, 2000.0,  50.0),
        ('kcat',    'γ_A (1/s)',                         0.05,    1.0,   0.25,   0.01),
        ('gammac',  'γ_H (×10⁻³ 1/s)',                  0.5,   10.0,   3.072,   0.1,  1e3),
        ('n',       'n (Hill coefficient)',               1.0,    5.0,    3.0,   1.0),
    ],
    dashboard_name='ABA Biosynthesis'
)
dashboard_aba


Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=30.0, name='V_stress (nM/s)', start=1.0, value=8.85, width=400)
            [1] FloatSlider(end=200.0, name='V₊ (nM/s)', start=10.0, step=1.0, value=67.58, width=400)
            [2] FloatSlider(end=10.0, name='V₋ (nM/s)', start=0.1, step=0.05, value=1.51, width=400)
            [3] FloatSlider(end=5000.0, name='K₊ = K₋ (nM)', start=200.0, step=20.0, value=1560.0, width=400)
        [1] Column
            [0] FloatSlider(end=5000.0, name='K_d (nM)', start=500.0, step=50.0, value=2000.0, width=400)
            [1] FloatSlider(name='γ_A (1/s)', start=0.05, step=0.01, value=0.25, width=400)
            [2] FloatSlider(end=10.0, name='γ_H (×10⁻³ 1/s)', start=0.5, value=3.072, width=400)
            [3] FloatSlider(end=5.0, name='n (Hill coefficient)', start=1.0, step=1.0, value=3.0, width=400)
    [2] Row
        [0] Button(button_type='success', name='Compute', width=200)
        [1] Button(button_type='warning', name='Reset Defaults', width=200)
    [3] StaticText(value='Ready. Adjust s...)
    [4] Bokeh(Row)

---
## Dashboard 4: ABA Signaling Cascade

These parameters govern the **receptor-kinase-phosphatase cascade** (SI Section S3.2, Eqs. S28, S33) that translates cytosolic ABA into SLAC1 ion channel activation via the OST1 bicyclic futile cycle with non-competitive PP2C inhibition.

In [15]:
dashboard_signaling = make_dashboard(
    title='ABA Signaling Cascade',
    description='Receptor binding, kinase/phosphatase rates, Michaelis constants, and protein levels.',
    slider_specs=[
        # Kinase / phosphatase rates
        # key    label                              min     max      default    step    [scale]
        ('k1',   'k₁ MAP3K→OST1 (1/s)',           0.005,   0.1,    0.03,     0.005),
        ('k2',   'k₂ PP2C→OST1 (1/s)',            0.005,   0.1,    0.033,    0.001),
        ('k3',   'k₃ OST1*→SLAC1 (1/s)',          0.5,    10.0,    3.75,     0.25),
        ('k4',   'k₄ PP2C→SLAC1 (1/s)',           0.02,    0.5,    0.139,    0.005),
        # Michaelis constants
        ('Km1',  'K_m1 OST1 phosph. (nM)',        500.0, 8000.0, 2500.0,   100.0),
        ('Km2',  'K_m2 OST1 dephosph. (nM)',       10.0,  500.0,   89.7,     5.0),
        ('Km3',  'K_m3 SLAC1 phosph. (µM)',         1.0,   50.0,   18.93,    0.5,  1e-3),
        ('Km4',  'K_m4 SLAC1 dephosph. (nM)',      50.0, 2000.0,  597.0,    10.0),
        # Receptor / inhibition
        ('KD',   'K_D ABA–PYR dissoc. (nM)',      200.0, 3000.0, 1000.0,    50.0),
        ('KI',   'K_I PP2C inhibition (nM)',         5.0,  100.0,   30.0,     1.0),
        # Protein totals
        ('OST1T',  '[OST1]_T (nM)',               200.0, 3000.0, 1000.0,   50.0),
        ('PP2CT',  '[PP2C]_T (nM)',               200.0, 3000.0, 1000.0,   50.0),
        ('SLAC1T', '[SLAC1]_T (nM)',              200.0, 3000.0, 1000.0,   50.0),
        ('MAP3KT',  '[MAP3K]_T (nM)',              200.0, 3000.0, 1000.0,   50.0),
    ],
    dashboard_name='ABA Signaling'
)
dashboard_signaling

Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=0.1, name='k₁ MAP3K→OST1 (1/s)', start=0.005, step=0.005, value=0.03, width=400)
            [1] FloatSlider(end=0.1, name='k₂ PP2C→OST1 (1/s)', start=0.005, step=0.001, value=0.033, width=400)
            [2] FloatSlider(end=10.0, name='k₃ OST1*→SLAC1 (1/s)', start=0.5, step=0.25, value=3.75, width=400)
            [3] FloatSlider(end=0.5, name='k₄ PP2C→SLAC1 (1/s)', start=0.02, step=0.005, value=0.139, width=400)
            [4] FloatSlider(end=8000.0, name='K_m1 OST1 phosph. (nM)', start=500.0, step=100.0, value=2500.0, width=400)
            [5] FloatSlider(end=500.0, name='K_m2 OST1 dephosph. (..., start=10.0, step=5.0, value=89.7, width=400)
            [6] FloatSlider(end=50.0, name='K_m3 SLAC1 phosph. (µM)', start=1.0, step=0.5, value=18.93, width=400)
        [1] Column
            [0] FloatSlider(end=2000.0, name='K_m4 SLAC1 dephosph. (..., start=50.0, step=10.0, value=597.0, width=400)
            [1] FloatSlider(end=3000.0, name='K_D ABA–PYR d..., start=200.0, step=50.0, value=1000.0, width=400)
            [2] FloatSlider(end=100.0, name='K_I PP2C inhibition (..., start=5.0, step=1.0, value=30.0, width=400)
            [3] FloatSlider(end=3000.0, name='[OST1]_T (nM)', start=200.0, step=50.0, value=1000.0, width=400)
            [4] FloatSlider(end=3000.0, name='[PP2C]_T (nM)', start=200.0, step=50.0, value=1000.0, width=400)
            [5] FloatSlider(end=3000.0, name='[SLAC1]_T (nM)', start=200.0, step=50.0, value=1000.0, width=400)
            [6] FloatSlider(end=3000.0, name='[MAP3K]_T (nM)', start=200.0, step=50.0, value=1000.0, width=400)
    [2] Row
        [0] Button(button_type='success', name='Compute', width=200)
        [1] Button(button_type='warning', name='Reset Defaults', width=200)
    [3] StaticText(value='Ready. Adjust s...)
    [4] Bokeh(Row)

---
## Dashboard 5: Electrophysiology

These parameters control **ion channels and pumps**, determining the electrical properties and ion fluxes in the guard cell.

In [16]:
dashboard_electrophys = make_dashboard(
    title='Electrophysiology',
    description='Ion concentrations and ABA-pump sensitivity governing guard cell membrane potential.',
    slider_specs=[
        # key                label                   min    max    default  step
        ('Clin0',           'Cl⁻_in (mM)',          50.0, 500.0,  260.0,  10.0),
        ('Clout0',          'Cl⁻_out (mM)',          5.0, 100.0,   22.0,   1.0),
        ('Kin0',            'K⁺_in (mM)',           50.0, 500.0,  210.0,  10.0),
        ('Kout0',           'K⁺_out (mM)',           5.0, 100.0,   20.0,   1.0),
        ('ABA_Hpump_Khalf', 'K_ABA (nM)',           10.0, 200.0,   50.0,   5.0),
        # --- GORK channel ---
        ('delta_GORK',       'δ_GORK',               0.5,   5.0,    2.0,   0.1),
        ('V1by2_Kconc_GORK', 'V½_GORK K⁺ (mM)',     10.0, 500.0,  150.0,  10.0),
        # --- KAT1 channel ---
        ('delta_KAT1',       'δ_KAT1',               0.5,   5.0,    1.6,   0.1),
        ('V1by2_KAT1',       'V½_KAT1 (mV)',       -300.0, -50.0, -170.0,   5.0, 1000.0),
        # --- H⁺-ATPase pump ---
        ('DeltaGATP',        'ΔG_ATP (kJ/mol)',     -40.0, -15.0,  -26.0,   0.5, 0.001),
        ('E0pump',           'E₀_pump (×10⁶)',        0.1,   5.0,  1.1033,  0.0001, 1e-6),
        ('k1pump',           'k₁_pump',              0.01,   2.0,   0.55,  0.01),
        ('k2pump',           'k₂_pump (×10⁻⁴)',      0.1,  10.0,   2.58,   0.01, 1e4),
        # --- Symporter / SLAC1 scaling ---
        ('V0Cl',             'V₀_Cl (×10⁴)',         1.0,  30.0,   9.25,  0.25, 1e-4),
        # --- ABA pump inhibition ---
        ('ABA_Hpump_n',      'n_ABA pump',            0.5,   4.0,    1.0,   0.5),
    ],
    dashboard_name='Electrophysiology'
)
dashboard_electrophys


Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=500.0, name='Cl⁻_in (mM)', start=50.0, step=10.0, value=260.0, width=400)
            [1] FloatSlider(end=100.0, name='Cl⁻_out (mM)', start=5.0, step=1.0, value=22.0, width=400)
            [2] FloatSlider(end=500.0, name='K⁺_in (mM)', start=50.0, step=10.0, value=210.0, width=400)
            [3] FloatSlider(end=100.0, name='K⁺_out (mM)', start=5.0, step=1.0, value=20.0, width=400)
            [4] FloatSlider(end=200.0, name='K_ABA (nM)', start=10.0, step=5.0, value=50.0, width=400)
            [5] FloatSlider(end=5.0, name='δ_GORK', start=0.5, value=2.0, width=400)
            [6] FloatSlider(end=500.0, name='V½_GORK K⁺ (mM)', start=10.0, step=10.0, value=150.0, width=400)
            [7] FloatSlider(end=5.0, name='δ_KAT1', start=0.5, value=1.6, width=400)
        [1] Column
            [0] FloatSlider(end=-50.0, name='V½_KAT1 (mV)', start=-300.0, step=5.0, value=-170.0, width=400)
            [1] FloatSlider(end=-15.0, name='ΔG_ATP (kJ/mol)', start=-40.0, step=0.5, value=-26.0, width=400)
            [2] FloatSlider(end=5.0, name='E₀_pump (×10⁶)', start=0.1, step=0.0001, value=1.1033, width=400)
            [3] FloatSlider(end=2.0, name='k₁_pump', start=0.01, step=0.01, value=0.55, width=400)
            [4] FloatSlider(end=10.0, name='k₂_pump (×10⁻⁴)', start=0.1, step=0.01, value=2.58, width=400)
            [5] FloatSlider(end=30.0, name='V₀_Cl (×10⁴)', start=1.0, step=0.25, value=9.25, width=400)
            [6] FloatSlider(end=4.0, name='n_ABA pump', start=0.5, step=0.5, value=1.0, width=400)
    [2] Row
        [0] Button(button_type='success', name='Compute', width=200)
        [1] Button(button_type='warning', name='Reset Defaults', width=200)
    [3] StaticText(value='Ready. Adjust s...)
    [4] Bokeh(Row)

In [ ]:
# Build the page that will be served by panel convert.
pn.Column(
    "# Stomatal transients dashboard",
    "VPD step-response transients (80% → 60% RH) with experimental data overlay (Merilo et al. 2018).",
    "**Figure 2** of Desai & Stroock (2025), *bioRxiv* — "
    "[doi.org/10.64898/2025.12.26.696581](https://doi.org/10.64898/2025.12.26.696581)"
    " · [← All dashboards](./index.html)",
    pn.Tabs(
        ("Hydraulics",        dashboard_hydraulics),
        ("V_stress",          dashboard_vstress),
        ("ABA biosynthesis",  dashboard_aba),
        ("ABA signaling",     dashboard_signaling),
        ("Electrophysiology", dashboard_electrophys),
    ),
).servable()
